# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata['name']}")
print(f"Dataset description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset includes tabular data from three patients. The Croissant schema defines record sets corresponding to structured tables. Each entity in the dataset is referenced by its unique `@id`.

First, let's enumerate all available record sets and their `@id`s.

In [ ]:
# List all record sets defined in the Croissant schema
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets detected. Please check the schema or the mlcroissant library version.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")

Now examine fields (columns) for each record set. We use `mlcroissant` to enumerate fields based on their `@id`s.

In [ ]:
# Show columns (fields) for each record set by @id
for rs in record_sets:
    fields = rs.get('fields', [])
    print(f"Record Set @id: {rs['@id']} - {rs.get('name', 'N/A')}")
    print("Fields:")
    for fld in fields:
        # Each field is a dict with @id
        print(f"  - @id: {fld['@id']}, name: {fld.get('name', 'N/A')} (type: {fld.get('dataType', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's load all available record sets by their `@id` into Pandas DataFrames.

In [ ]:
# Prepare to extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set '@id': {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head(), "\n")
        else:
            print(f"No records found for record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Error loading record set '@id': {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes using field `@id`. 

**Note:** Field operations reference columns by their `@id` from schema, not by display names.

Here is an example for the first record set, if present.

In [ ]:
# Choose a record set for demo (first available) and a numeric field
if dataframes:
    demo_record_set_id = list(dataframes.keys())[0]
    df = dataframes[demo_record_set_id]

    # Find a numeric field from the record set schema
    demo_record_set = [rs for rs in record_sets if rs['@id'] == demo_record_set_id][0]
    numeric_fields = [f for f in demo_record_set.get('fields', []) if f.get('dataType', '').lower() in ['integer', 'float', 'number']]

    if numeric_fields:
        numeric_field = numeric_fields[0]['@id']
        print(f"Using numeric field: {numeric_field}")
        
        # Basic filtering: filter values above mean
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in '{demo_record_set_id}' with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field for filtered records
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # If there's a categorical field, group by it
        categorical_fields = [f for f in demo_record_set.get('fields', []) if f.get('dataType', '').lower()=='text']
        if categorical_fields:
            group_field = categorical_fields[0]['@id']
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by '{group_field}':")
                print(grouped_df.head())
    else:
        print(f"No numeric fields found in record set '@id': {demo_record_set_id}.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using record set and field `@id`s. Here, an example histogram and bar plot for numeric and categorical fields are shown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field, bar plot for categorical grouping
if dataframes and 'filtered_df' in locals():
    if numeric_field in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[numeric_field], bins=5, kde=True)
        plt.title(f"Distribution of {numeric_field} (filtered)")
        plt.xlabel(numeric_field)
        plt.show()

    # Categorical plot
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} mean by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No data available for visualization. Run EDA cell first.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`. We reviewed the available record sets and fields, loaded the tabular data, and performed basic exploratory and visualization steps. Please refer to the Croissant schema and the dataset description for further info, especially regarding data limitations due to the small cohort size and clinical scope.